# Cool places pipeline: CSV to GeoJSON

This notebook adds a fifth layer to CatMap: the City of Montréal's **air-conditioned public places** (lieux publics climatisés), where anyone can go to cool off for free during a heat wave: libraries, community centres, arenas, borough offices and so on.

```
data/lieux_publics_climatises.csv  input: the City of Montréal's open data
data/cool_places.geojson           output, written by this notebook
site/app.js                        draws it as purple dots (off until you tick it in the layer switcher)
```

The CSV has `long` and `lat` columns in degrees, so this follows Lab 4's **Recipe 1**. Run each cell in order, top to bottom.

## 1) Import libraries and load the CSV
The phone columns are read as text (`dtype=str`); otherwise pandas turns `5143676363` into the number `5143676363.0`.

In [ ]:
import os
import pandas as pd
import geopandas as gpd

in_path = "../data/lieux_publics_climatises.csv"
df = pd.read_csv(in_path, dtype={"telephone": str, "telephone_numero_poste": str})
print(len(df), "cool places in", df["arrdondissement"].nunique(), "boroughs")
print(df.columns.tolist())

## 2) Drop places without coordinates, make points
Points need longitude first (x), then latitude (y), in `EPSG:4326`.

In [ ]:
before = len(df)
df = df.dropna(subset=["lat", "long"])
print("before:", before, "after:", len(df))

gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df["long"], df["lat"]), crs="EPSG:4326")
gdf.geometry.head()

## 3) Keep only what the popups show, then write
Four columns with short English names. The phone number becomes `514-367-6363`, with the extension added as `poste 4164` when there is one. The `accessibilite` column is a long list of features; we keep only whether it says `Accessible en fauteuil roulant` (wheelchair accessible). Note the city's column is spelled `arrdondissement`, with an extra *d*. **Output:** `../data/cool_places.geojson`

In [ ]:
def format_phone(number, ext):
    # "5143676363", "4164" -> "514-367-6363 poste 4164"; blank stays blank
    if pd.isna(number):
        return None
    text = f"{number[:3]}-{number[3:6]}-{number[6:]}"
    return f"{text} poste {ext}" if pd.notna(ext) else text

keep = gpd.GeoDataFrame({
    "name": gdf["nom"],
    "borough": gdf["arrdondissement"],
    "address": gdf["adresse_principale"],
    "phone": [format_phone(n, e) for n, e in zip(gdf["telephone"], gdf["telephone_numero_poste"])],
    "wheelchair": gdf["accessibilite"].fillna("").str.contains("Accessible en fauteuil roulant", regex=False),
}, geometry=gdf.geometry, crs=gdf.crs)

out_path = "../data/cool_places.geojson"
keep.to_file(out_path, driver="GeoJSON", COORDINATE_PRECISION=6)
print("Wrote", out_path, "-", round(os.path.getsize(out_path) / 1_000_000, 3), "MB")
keep.head()

## 4) Check the file
The check cell from Lab 4, with the four settings changed for this file.

In [ ]:
# ---- settings: change these for your own data ----
FILE = "../data/cool_places.geojson"
WEST, EAST, SOUTH, NORTH = -73.98, -73.47, 45.40, 45.71   # a box around the island of Montréal, in degrees
EXPECTED = 222            # how many features you expect (rows after your filters)
POPUP_FIELDS = ["name", "borough", "address", "phone", "wheelchair"]
# ----------------------------------------------------

back = gpd.read_file(FILE)
problems = []
if len(back) != EXPECTED:
    problems.append(f"{len(back)} features, but you expected {EXPECTED}.")
if back.crs is None or back.crs.to_epsg() != 4326:
    problems.append(f"The file is in {back.crs}, not longitude and latitude (EPSG:4326).")
minx, miny, maxx, maxy = back.total_bounds
if not (WEST <= minx and maxx <= EAST and SOUTH <= miny and maxy <= NORTH):
    problems.append(f"Some features are outside your box: x {minx:.3f} to {maxx:.3f}, y {miny:.3f} to {maxy:.3f}.")
missing = [f for f in POPUP_FIELDS if f not in back.columns]
if missing:
    problems.append(f"Missing popup fields: {missing}.")
mb = os.path.getsize(FILE) / 1_000_000
if mb > 5:
    problems.append(f"The file is {mb:.1f} MB; over about 5 MB the page gets slow. Filter or simplify.")
print("Geometry types:", sorted(back.geom_type.unique()))
print("All checks passed" if not problems else "Not yet:\n- " + "\n- ".join(problems))

## 5) Next: open the map
1. In VS Code, right-click `site/index.html` and choose Open with Live Server.
2. The cool places start hidden: tick **Free AC (cool places)** in the layer switcher (top right) to show them as purple dots.